# WoundWatch — Gemma 4 E4B Fine-tuning with Unsloth

**Kaggle Notebook 실행 방법:**
1. 우측 Settings → Accelerator: GPU T4 x2 (또는 P100)
2. Add Data → `laithjj/diabetic-foot-ulcer-dfu` 추가
3. Add Data → `leoscode/wound-segmentation-images` 추가
4. Secrets → `HF_TOKEN` 추가 (HuggingFace 토큰)
5. Run All

**목표:** Gemma 4 E4B 멀티모달 모델을 당뇨발 궤양 이미지 분석 태스크에 파인튜닝

In [ ]:
# ── 1. 의존성 설치 ────────────────────────────────────────────────────────────
import subprocess
subprocess.run([
    "pip", "install", "--quiet",
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
], check=True)
subprocess.run([
    "pip", "install", "--quiet", "--no-deps",
    "trl", "peft", "accelerate", "bitsandbytes"
], check=True)
print('설치 완료')

In [ ]:
# ── 2. 라이브러리 임포트 ──────────────────────────────────────────────────────
import os
import json
import random
from pathlib import Path

import numpy as np
import cv2
from PIL import Image
from datasets import Dataset
from kaggle_secrets import UserSecretsClient

from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

random.seed(42)
print('임포트 완료')

In [ ]:
# ── 3. 데이터셋 경로 & 구조 확인 ─────────────────────────────────────────────
DFU_DIR = Path("/kaggle/input/diabetic-foot-ulcer-dfu")
SEG_DIR = Path("/kaggle/input/wound-segmentation-images")

print("=== DFU Binary Dataset ===")
for p in sorted(DFU_DIR.iterdir())[:20]:
    print(f"  {p.relative_to(DFU_DIR)}")

print("\n=== Segmentation Dataset ===")
for p in sorted(SEG_DIR.iterdir())[:20]:
    print(f"  {p.relative_to(SEG_DIR)}")

In [ ]:
# ── 4. 데이터 전처리 ──────────────────────────────────────────────────────────
ULCER_KEYWORDS   = {"ulcer", "dfu", "wound", "positive", "diabetic"}
HEALTHY_KEYWORDS = {"healthy", "normal", "non_dfu", "negative", "control"}

PROMPT = (
    "Analyze this diabetic foot image. "
    "Determine: (1) wound presence, "
    "(2) wound coverage percentage, "
    "(3) severity score 0–10, "
    "(4) recommended action."
)


def area_to_severity(ratio: float) -> int:
    if ratio < 0.02: return 3
    if ratio < 0.05: return 5
    if ratio < 0.10: return 7
    return 9


def make_sample(img_path: str, is_ulcer: bool, wound_ratio: float = 0.0) -> dict:
    if is_ulcer:
        sev = area_to_severity(wound_ratio)
        response = (
            f"Diabetic foot ulcer detected. "
            f"Wound coverage: {wound_ratio:.1%}. "
            f"Severity: {sev}/10. "
            f"{'Immediate' if sev >= 7 else 'Urgent'} clinical evaluation recommended."
        )
    else:
        response = (
            "No wound detected. "
            "Foot appears healthy with intact skin. "
            "Severity: 0/10. Continue routine monitoring."
        )
    return {
        "image_path": img_path,
        "response": response,
        "is_ulcer": is_ulcer,
    }


samples = []

# Dataset 1: 폴더 기반 이진 분류
for folder in DFU_DIR.rglob("*"):
    if not folder.is_dir():
        continue
    name = folder.name.lower()
    if any(k in name for k in ULCER_KEYWORDS):
        label = True
    elif any(k in name for k in HEALTHY_KEYWORDS):
        label = False
    else:
        continue
    for img in folder.glob("*.jpg"):
        samples.append(make_sample(str(img), label))
    for img in folder.glob("*.png"):
        samples.append(make_sample(str(img), label))

print(f"DFU binary: {len(samples)}개")

# Dataset 2: 세그멘테이션 마스크
seg_image_dir = next((d for d in SEG_DIR.rglob("images") if d.is_dir()), None)
seg_mask_dir  = next((d for d in SEG_DIR.rglob("masks")  if d.is_dir()), None)

seg_count = 0
if seg_image_dir and seg_mask_dir:
    for img_path in sorted(seg_image_dir.glob("*.jpg")):
        mask_path = seg_mask_dir / f"{img_path.stem}.png"
        if mask_path.exists():
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
            ratio = np.count_nonzero(binary) / binary.size
        else:
            ratio = 0.0
        samples.append(make_sample(str(img_path), is_ulcer=True, wound_ratio=ratio))
        seg_count += 1

print(f"Segmentation: {seg_count}개")
print(f"총 샘플: {len(samples)}개")

random.shuffle(samples)
split = int(len(samples) * 0.85)
train_samples = samples[:split]
val_samples   = samples[split:]
print(f"Train: {len(train_samples)} / Val: {len(val_samples)}")

In [ ]:
# ── 5. HuggingFace Dataset 변환 ───────────────────────────────────────────────
# PIL Image를 content dict 안에 직접 넣으면 Dataset 직렬화 시 손상됨
# 해결: "file://절대경로" 문자열로 넣으면 process_vision_info가 직접 로드함

def to_hf_format(sample: dict) -> dict:
    abs_path = str(Path(sample["image_path"]).resolve())
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": f"file://{abs_path}"},
                {"type": "text",  "text": PROMPT},
            ],
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": sample["response"]}],
        },
    ]
    return {"messages": messages}


train_dataset = Dataset.from_list([to_hf_format(s) for s in train_samples])
val_dataset   = Dataset.from_list([to_hf_format(s) for s in val_samples])

print(f"HF Dataset 준비 완료: train={len(train_dataset)}, val={len(val_dataset)}")
print("샘플 경로 확인:", train_dataset[0]["messages"][0]["content"][0]["image"][:60])

In [ ]:
# ── 6. Gemma 4 E4B 모델 로드 + LoRA 설정 ─────────────────────────────────────
# 한 셀에서 실행 — 재실행해도 모델을 새로 불러오므로 LoRA 중복 에러 없음
model, processor = FastVisionModel.from_pretrained(
    "unsloth/gemma-4-E4B-it",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=42,
)
model.print_trainable_parameters()

In [ ]:
# ── 7. LoRA 설정 ──────────────────────────────────────────────────────────────
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=42,
)
model.print_trainable_parameters()

In [ ]:
# ── 8. Trainer 설정 ───────────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    tokenizer=processor,
    data_collator=UnslothVisionDataCollator(model, processor),
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        fp16=True,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        output_dir="/kaggle/working/woundwatch-checkpoints",
        report_to="none",
        remove_unused_columns=False,
        dataset_kwargs={"skip_prepare_dataset": True},
    ),
)
print("Trainer 준비 완료")

In [ ]:
# ── 9. 파인튜닝 실행 ──────────────────────────────────────────────────────────
trainer_stats = trainer.train()
print(f"\n훈련 완료")
print(f"총 스텝: {trainer_stats.global_step}")
print(f"최종 Loss: {trainer_stats.training_loss:.4f}")

In [ ]:
# ── 8. 추론 테스트 ───────────────────────────────────────────────────────────
FastVisionModel.for_inference(model)

test_img_path = train_samples[0]["image_path"]
test_image = Image.open(test_img_path).convert("RGB")

# Gemma4Processor는 content 안 이미지를 자동으로 추출함
# images= 를 따로 넘기면 중복 충돌 → 넣지 않음
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": test_image},
            {"type": "text",  "text": PROMPT},
        ],
    }
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=False,   # temperature 제거 → greedy decoding
)
result = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print("=== 추론 결과 ===")
print(result)

In [ ]:
# ── 11. HuggingFace Hub 업로드 ────────────────────────────────────────────────
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

# HF 사용자명을 본인 아이디로 변경하세요
HF_REPO = "your-hf-username/woundwatch-gemma4-e4b"

model.push_to_hub(HF_REPO, token=hf_token)
processor.push_to_hub(HF_REPO, token=hf_token)

print(f"\n업로드 완료: https://huggingface.co/{HF_REPO}")

## 다음 단계

업로드된 모델은 FastAPI 백엔드에서 이렇게 불러와요:

```python
from unsloth import FastVisionModel

model, processor = FastVisionModel.from_pretrained(
    "your-hf-username/woundwatch-gemma4-e4b",
    load_in_4bit=True,
)
FastVisionModel.for_inference(model)
```

또는 Ollama 로컬 실행을 위해 GGUF 변환:
```bash
ollama create woundwatch -f Modelfile
```